## Analysis after dimensionality reduction

After multiple rounds of data cleaning, the improvement in the baseline model's silhouette score reached a bottleneck. To further mitigate the "curse of dimensionality" inherent in high-dimensional sparse features and to enhance the model's sensitivity to core semantics, I explored the integration of dimensionality reduction techniques. Specifically, I applied Latent Semantic Analysis (LSA via TruncatedSVD) combined with L2 normalization for the BoW and TF-IDF models, and Principal Component Analysis (PCA) with L2 normalization for the SBERT dense vectors.

Experimental results demonstrated that compressing the SBERT embeddings from 384 to 80 dimensions yielded a significant improvement in the clustering silhouette score. However, the PCA output revealed that this reduction process resulted in a loss of approximately 15% of the explained variance. Given the context of our customer support ticket data, it remains uncertain whether these 15% marginal features contain critical, minority-class customer complaints. Therefore, to prioritize model robustness and preserve information integrity, this aggressive dimensionality reduction approach was excluded from our primary analytical pipeline.

Additionally, I tried UMAP as a non-linear dimensionality reduction technique. Although the UMAP-transformed data exhibited an artificially inflated and dramatic surge in silhouette scores, I ultimately chose to discard this method. The fundamental reason is that UMAP forcibly distorts the global topological structure to cluster local neighbors together, which inherently destroys the authentic Euclidean distance metrics. Calculating distance-dependent silhouette scores within this distorted space generates a severe "mathematical illusion" akin to overfitting, rendering the results entirely meaningless for guiding real-world business classification.

This code snippet is an optimization added to the existing workflow. It depends on the original environment and data and cannot run independently. If needed, please add it to the complete code for testing.

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
import umap

### LSA & Normalization for BOA & TF-IDF

In [ ]:
print("LSA (TruncatedSVD) + L2 Normalization")

def optimize_sparse_matrix(X, name, n_components=80):
    print(f"\n Optimizing {name}...")
    # TruncatedSVD 
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    X_lsa = svd.fit_transform(X)
    print(f"  LSA explained variance ({n_components} components): {sum(svd.explained_variance_ratio_):.4f}")
    
    # L2 Normalization
    X_improved = normalize(X_lsa, norm='l2')
    return X_improved

In [ ]:
# x_boa
X_boa_improved = optimize_sparse_matrix(X_boa, "BoW (Raw)")

# Against Ticket Type
clustering_pipeline(X=X_boa_improved, 
                    y=ticket_type_encoded, 
                    cluster_range=range(2, 10), 
                    evaluation_y="Ticket Type (Improved BoW)", 
                    metric='euclidean'
                    )

# Against Ticket Subject
clustering_pipeline(X=X_boa_improved, 
                    y=ticket_subj_encoded, 
                    cluster_range=range(2, 10), 
                    evaluation_y="Ticket Subject (Improved BoW)", 
                    metric='euclidean'
                    )

In [ ]:
# x_boa_cust
X_boa_cust_improved = optimize_sparse_matrix(X_boa_cust, "BoW (Custom Stopwords)")

# Against Ticket Type
clustering_pipeline(X=X_boa_cust_improved, 
                    y=ticket_type_encoded, 
                    cluster_range=range(2, 10), 
                    evaluation_y="Ticket Type (Improved BoW Cust)", 
                    metric='euclidean'
                    )

# Against Ticket Subject
clustering_pipeline(X=X_boa_cust_improved, 
                    y=ticket_subj_encoded, 
                    cluster_range=range(2, 10), 
                    evaluation_y="Ticket Subject (Improved BoW Cust)", 
                    metric='euclidean'
                    )

In [ ]:
# x_tfidf
X_tfidf_improved = optimize_sparse_matrix(X_tfidf, "TF-IDF (Raw)")

# Against Ticket Type
clustering_pipeline(X=X_tfidf_improved, 
                    y=ticket_type_encoded, 
                    cluster_range=range(2, 10), 
                    evaluation_y="Ticket Type (Improved TF-IDF)", 
                    metric='euclidean'
                    )

# Against Ticket Subject
clustering_pipeline(X=X_tfidf_improved, 
                    y=ticket_subj_encoded, 
                    cluster_range=range(2, 10), 
                    evaluation_y="Ticket Subject (Improved TF-IDF)", 
                    metric='euclidean'
                    )

In [ ]:
# x_tfidf_cust
X_tfidf_cust_improved = optimize_sparse_matrix(X_tfidf_cust, "TF-IDF (Custom Stopwords)")

# Against Ticket Type
clustering_pipeline(X=X_tfidf_cust_improved, 
                    y=ticket_type_encoded, 
                    cluster_range=range(2, 10), 
                    evaluation_y="Ticket Type (Improved TF-IDF Cust)", 
                    metric='euclidean'
                    )

# Against Ticket Subject
clustering_pipeline(X=X_tfidf_cust_improved, 
                    y=ticket_subj_encoded, 
                    cluster_range=range(2, 10), 
                    evaluation_y="Ticket Subject (Improved TF-IDF Cust)", 
                    metric='euclidean'
                    )

### PCA + Normalization

In [ ]:
print("PCA + L2 Normalization")

# 1. L2 Normalization: Transforms Euclidean distance into the equivalent space of cosine similarity.
X_sbert_norm = normalize(X_sbert, norm='l2')

# 2. PCA Dimensionality Reduction: Reduce the curse of dimensionality and extract core features (from 384D to 80D)
pca_opt = PCA(n_components=80, random_state=42)
X_sbert_improved = pca_opt.fit_transform(X_sbert_norm)

print(f"PCA explained variance ratio (20 components): {sum(pca_opt.explained_variance_ratio_):.4f}")

In [ ]:
# SBERT_cust
# Against Ticket Type
clustering_pipeline(
    X=X_sbert_improved,
    y=ticket_type_encoded,
    cluster_range=range(2, 15),
    evaluation_y="Ticket Type (Improved SBERT)",
    metric='euclidean'  
)

# Against Ticket Subject
clustering_pipeline(
    X=X_sbert_improved,
    y=ticket_subj_encoded,
    cluster_range=range(2, 15),
    evaluation_y="Ticket Subject (Improved SBERT)",
    metric='euclidean'  
)

### Umap

In [ ]:
# UMAP
umap_reducer = umap.UMAP(n_neighbors=15, min_dist=0.0, n_components=10, metric='cosine', random_state=42)
X_sbert_umap = umap_reducer.fit_transform(X_sbert)

In [ ]:
# SBERT_cust
# Against Ticket Type
print("\n--- Running UMAP against Ticket Type ---")
clustering_pipeline(
    X=X_sbert_umap,
    y=ticket_type_encoded,
    cluster_range=range(2, 10),
    evaluation_y="Ticket Type (UMAP)",
    metric='euclidean'  
)

# Against Ticket Subject
print("\n--- Running UMAP against Ticket Subject ---")
clustering_pipeline(
    X=X_sbert_umap,
    y=ticket_subj_encoded,
    cluster_range=range(2, 10),
    evaluation_y="Ticket Subject (UMAP)",
    metric='euclidean'  
)